In [1]:
%%bash
git clone https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
cd /kaggle/working/sd-scripts

pip install --upgrade pip
pip install -r requirements.txt
pip install xformers bitsandbytes accelerate prodigyopt

accelerate config default

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Ignoring tensorboardX: markers 'platform_machine == "ARM64"' don't match your environment
Obtaining file:///kaggle/working/sd-scripts (from -r requirements.txt (line 58))
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cloning into '/kaggle/working/sd-scripts'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.1.0 which is incompatible.


In [2]:
# Cell 2: Multi-Angle Dataset Synthesis & Pipeline
import os
import glob
from PIL import Image

# Corrected path to Mia's dataset
input_dir = "/kaggle/input/datasets/anindoakb/mialora"
output_dir = "/kaggle/working/train_mia/img/10_mia_mlbb"
os.makedirs(output_dir, exist_ok=True)

image_paths = glob.glob(os.path.join(input_dir, "*.*"))

for idx, img_path in enumerate(image_paths):
    try:
        img = Image.open(img_path).convert("RGB")
        base_out = os.path.join(output_dir, f"img_{idx:04d}.jpg")
        img.save(base_out, "JPEG", quality=100)
        
        img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
        flip_out = os.path.join(output_dir, f"img_{idx:04d}_aug_flip.jpg")
        img_flip.save(flip_out, "JPEG", quality=100)
    except Exception as e:
        pass

print(f"Dataset synthesis complete. Images: {len(os.listdir(output_dir))}")

Dataset synthesis complete. Images: 92


In [3]:
# Cell 3: Mia Sample Prompts
import os

prompt_file = "/kaggle/working/prompt.txt"

# Updated to use Mia's trigger tag and archer traits
prompts = [
    "mia_mlbb, 1girl, drawing bow, glowing magic arrow, intense gaze, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "mia_mlbb, 1girl, jumping mid-air, dynamic archery pose, flowing hair, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution",
    "mia_mlbb, 1girl, standing in enchanted forest, moonlight, elegant archer dress, masterpiece, best quality, ultra-detailed anime illustration"
]

flags = "--w 768 --h 1024 --l 7.5 --s 30"

with open(prompt_file, "w", encoding="utf-8") as f:
    for prompt in prompts:
        f.write(f"{prompt} {flags}\n")

print(f"Generated sample prompts for Mia in {prompt_file}")

Generated sample prompts for Mia in /kaggle/working/prompt.txt


In [4]:
# Cell 4: Training Execution (Accidental delete command removed)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!cd /kaggle/working/sd-scripts && accelerate launch \
  --num_processes=1 \
  --mixed_precision="fp16" \
  train_network.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_mia/img" \
  --output_dir="/kaggle/working/output_mia" \
  --output_name="mia_lora" \
  --network_module="networks.lora" \
  --network_dim=256 \
  --network_alpha=128 \
  --optimizer_type="Prodigy" \
  --learning_rate=1.0 \
  --unet_lr=1.0 \
  --text_encoder_lr=1.0 \
  --lr_scheduler="cosine" \
  --optimizer_args weight_decay=0.01 d_coef=2 use_bias_correction=True safeguard_warmup=True \
  --min_snr_gamma=5.0 \
  --noise_offset=0.1 \
  --multires_noise_iterations=6 \
  --multires_noise_discount=0.3 \
  --network_dropout=0.1 \
  --scale_weight_norms=1.0 \
  --color_aug \
  --random_crop \
  --mixed_precision="fp16" \
  --save_precision="fp16" \
  --xformers \
  --gradient_checkpointing \
  --max_train_epochs=5 \
  --save_every_n_epochs=1 \
  --train_batch_size=2 \
  --resolution="512,512" \
  --save_model_as="safetensors"

/kaggle/working/sd-scripts/library/strategy_base.py:107: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/kaggle/working/sd-scripts/library/custom_train_functions.py:174: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/kaggle/working/sd-scripts/library/lpw_stable_diffusion.py:70: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
2026-10-03 20:29:11 INFO     Using v1 tokenizer                ]8;id=105213;file:///kaggle/working/sd-scripts/library/strategy_sd.py\strategy_sd.py]8;;\:]8;id=551094;file:///kaggle/working/sd-scripts/library/strategy_sd.py#34\34]8;;\
                    INFO     HTTP Request: GET                   ]8;id=886072;file:///usr/local/lib/python3.13/dist-packages/httpx/_client.py\_client.py]8;;\:]8;id=498127;file:///usr/local/lib/python3.13/dist-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/agent-ha                
              

In [5]:
# Cell 5: Exporting
import shutil
import os

output_folder = "/kaggle/working/output_mia"
archive_name = "/kaggle/working/mia_lora_outputs"

if os.path.exists(output_folder):
    shutil.make_archive(archive_name, 'zip', output_folder)
    print(f"Export complete. Archive saved to {archive_name}.zip")
else:
    print(f"Folder {output_folder} not found. Ensure training finished successfully.")

Export complete. Archive saved to /kaggle/working/mia_lora_outputs.zip
